In [1]:
import logging
from pathlib import Path
import sys 
import collections.abc

import enoslib as en

import tqdm
import pandas as pd 
import prometheus_pandas

[WARNING]: failed to patch stdout/stderr for fork-safety: 'OutStream' object
has no attribute 'buffer'
[WARNING]: failed to reconfigure stdout/stderr with the replace error handler:
'OutStream' object has no attribute 'reconfigure'


In [2]:
en.init_logging(level=logging.INFO)
en.check()



_____        ___  ____  _ _ _
 | ____|_ __  / _ \/ ___|| (_) |__
 |  _| | '_ \| | | \___ \| | | '_ \
 | |___| | | | |_| |___) | | | |_) |
 |_____|_| |_|\___/|____/|_|_|_.__/  9.2.0

 • Documentation: ]8;id=438746;https://discovery.gitlabpages.inria.fr/enoslib/\https://discovery.gitlabpages.inria.fr/enoslib/]8;;\                            
 • Source: ]8;id=868680;https://gitlab.inria.fr/discovery/enoslib\https://gitlab.inria.fr/discovery/enoslib]8;;\                                         
 • Chat: ]8;id=851787;https://framateam.org/enoslib\https://framateam.org/enoslib]8;;\

                         Dependency check                         
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Provider      ┃    Status     ┃ Hint                           ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Chameleon     │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ ChameleonKVM  │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ ChameleonEdge │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ Distem        │ NOT INSTALLED │ pip install enoslib[distem]    │
│ IOT-lab       │ NOT INSTALLED │ pip install enoslib[iotlab]    │
│ Grid'5000     │   INSTALLED   │                                │
│ Openstack     │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ Vagrant       │ NOT INSTALLED │ pip install enoslib[vagrant]   │
│ VMonG5k       │   INSTALLED   │                                │
└───────────────┴───────────────┴────────────────────────────────┘

ERROR    Unreachable hosts: [_AnsibleExecutionRecord(host='access.grid5000.fr',   ]8;id=402421;file:///home/guillaume/research/gepiciad/alloy-flink-tests/.venv/lib/python3.12/site-packages/enoslib/api.py\api.py]8;;\:]8;id=553112;file:///home/guillaume/research/gepiciad/alloy-flink-tests/.venv/lib/python3.12/site-packages/enoslib/api.py#1204\1204]8;;\
         status='UNREACHABLE', task='Connecting to                                           
         grosinosky@access.grid5000.fr', payload={'unreachable': True, 'msg':                
         "Failed to connect to the host via ssh: Warning: Permanently added                  
         'access.grid5000.fr' (ED25519) to the list of known                                 
         hosts.\r\ngrosinosky@access.grid5000.fr: Permission denied                          
         (publickey).", 'changed': False})]                                                  

                            Connectivity check                            
┏━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Provider  ┃ Key        ┃ Connectivity ┃ Hint                           ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Grid'5000 │ ssh:access │      ❌      │ Failed to connect to the host… │
│           │            │              │ grosinosky@access.grid5000.fr… │
│ VMonG5k   │ access     │      ❔      │ Check G5k status               │
└───────────┴────────────┴──────────────┴────────────────────────────────┘

## Docker Swarm init

In [4]:
# Install docker
def init_docker_swarm():
    registry_opts = dict(type="external", ip="docker-cache.grid5000.fr", port=80)
    d = en.Docker(
        agent=roles["control"], bind_var_docker="/tmp/docker", registry_opts=registry_opts,
        swarm=True,
        # Optional credentials for docker hub
        # credentials=dict(login="mylogin", password="mytoken"    
    )

    d.deploy()

[WARNING]: Invalid characters were found in group names but not replaced, use
-vvvv to see details


Output()

Finished 40 tasks (Gathering Facts,registry : include_tasks,registry : Installing 
dependencies,registry : Installing docker python bindings,registry : Creating docker state 
directory,registry : Bind mount the docker volume directory,registry : Installing latest 
docker,registry : Installing specific docker version,registry : Login to docker hub,registry 
: Detecting GPUs,registry : shell,registry : Installing nvidia-container-toolkit,registry : 
Create docker config directory,registry : Allow Docker to use an insecure registry,registry :
Restart docker daemon,swarm : include_tasks,swarm : Gather Facts,swarm : Start the 
manager,swarm : Create custom facts directory,swarm : Install custom facts,swarm : Loading 
facts,swarm : Join the swarm cluster) on {'gros-22.nancy.grid5000.fr', 
'gros-49.nancy.grid5000.fr', 'gros-32.nancy.grid5000.fr', 'gros-51.nancy.grid5000.fr', 
'gros-89.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

In [5]:

def init_docker_login(login, password):
    with en.play_on(roles=roles) as p: # read-only token, replace by yours
        p.raw(f"docker login -u {login} -p {password}")

Output()

Finished 1 tasks (raw) on {'gros-22.nancy.grid5000.fr', 'gros-49.nancy.grid5000.fr', 
'gros-32.nancy.grid5000.fr', 'gros-51.nancy.grid5000.fr', 'gros-89.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

In [6]:
def init_docker_nodes():
    with en.play_on(roles=roles, pattern_hosts=pattern_manager) as p:
        nodes = roles.get("injector")
        for node in nodes:
            p.raw(f"docker node update --label-add tier=injector {node.alias}")

        nodes = roles.get("kafka")
        for node in nodes:
            p.raw(f"docker node update --label-add tier=kafka {node.alias}")

        nodes = roles.get("jobmanager")
        for node in nodes:
            p.raw(f"docker node update --label-add tier=jobmanager {node.alias}")

        nodes = roles.get("taskmanager")
        for node in nodes:
            p.raw(f"docker node update --label-add tier=taskmanager {node.alias}")



Output()

Finished 1 tasks (raw) on {'gros-22.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

In [7]:
def init_docker_network():
    with en.play_on(roles=roles, pattern_hosts="control[0]") as p:
        p.docker_network(
            name="alloy-test_default",
            attachable=True,
            driver="overlay",
        )

Output()

Finished 1 tasks (docker_network) on {'gros-22.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

In [8]:
def init_monitoring():
    with en.play_on(roles=roles, pattern_hosts=pattern_manager) as t:
        t.copy(src="./deploy-monitoring.sh", dest="./", mode="0755")
        t.shell("./deploy-monitoring.sh")


Output()

Finished 2 tasks (copy,shell) on {'gros-22.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

## Functions

In [9]:
from time import sleep
import re
import datetime
# Function to extract service name

def extract_service_name_cpu(column_name):
    match = re.search(r'="[^_]+_(.+)"', column_name)
    # Regular expression to capture up to the second period
    pattern = r'^([^.]+\.[^.]+)\.'

    # Search for the pattern in the column name
    match = re.search(pattern, match.group(1))
    return "cpu" + "_" + match.group(1) if match else column_name

def deploy_swarm(compose_file="docker-compose.yml", name="test", env={}):
    print(f"Cleaning old swarm deployment {name}")

    with en.play_on(roles=roles, pattern_hosts=pattern_manager) as t:
        # Remove the old swarm stack
        t.shell(cmd=f"docker stack rm {name} || true")
        # Copy the entire overhead directory from the control machine to the target machine
        t.copy(src=f"../../{compose_file}", dest="./")
        # Deploy the new swarm stack
        env_str = " ".join("{}='{}'".format(*i) for i in env.items())
        command = f"{env_str} docker stack deploy --compose-file ./{compose_file} {name}"
        print(f"Launching new swarm deployment {name}: {command}")
        t.shell(cmd=command)

    wait = 30
    print(f"Wait for {wait} seconds")
    sleep(wait)     
def get_metrics_tunnel(min_date, max_date, metric_query="container_cpu_usage_seconds_total", step="5s", metric="cpu"):
    with en.G5kTunnel(roles["control"][0].address, 9090) as (local_address, local_port, _):
        from prometheus_pandas import query
        p = query.Prometheus(f"http://admin:prom-operator@{local_address}:{local_port}")    
        
        min_date_param = min_date.replace(microsecond=0).isoformat()+"Z"
        max_date_param = max_date.replace(microsecond=0).isoformat()+"Z"
        print (min_date_param + "-" + max_date_param)
        df_metric = p.query_range(
            metric_query,
            min_date.replace(microsecond=0).isoformat(),
            max_date.replace(microsecond=0).isoformat(),
            step
        )
        #df_metric = df_metric.rename(columns={df_metric.columns[0]: "ts"})
        # Rename columns
        df_metric = df_metric.rename(columns=extract_service_name_cpu)        
    return df_metric

In [10]:
def launch_publisher(host="kafka-edge1", compose_file="./docker-compose.berserker.yml", port="9092", topic="source", target_rps=1, replicas=1, key_name="id", key_value="identifier"):
    
    with en.play_on(roles=roles, pattern_hosts=pattern_manager) as t:
        # Remove the docker service
        t.shell(cmd="docker service rm test_berserker || true")
        t.copy(src=f"../../berserker", dest="./")
        t.copy(src=f"../../docker-compose.berserker.yml", dest="./")
        # Deploy the docker stack
        line = f"HOST={host} PORT={port} THROUGHPUT={target_rps} TOPIC={topic} REPLICAS={replicas} KEY_NAME={key_name} KEY_VALUE={key_value} docker stack deploy --compose-file {compose_file} test"
        print(line)
        t.shell(cmd=line)      

def stop_publisher():
    with en.play_on(roles=roles, pattern_hosts=pattern_manager) as t:
        t.shell(cmd="docker service rm test_berserker || true")


In [11]:
import requests
import os 

def deploy_job(filename, class_name, args=None):
    with en.G5kTunnel(roles["control"][0].address, 8081) as (local_address, local_port, _):
        print(f"{local_address}:{local_port}")
        res = requests.post(f"http://{local_address}:{local_port}/jars/upload", 
                            #headers={"Content-Type":},
                            #data={},
                            files={"jarfile": (
                                os.path.basename(filename),
                                open(filename, "rb"),
                                "application/x-java-archive"
                            )})
        
        jar_id = str(res.json()["filename"])
        jar_id = jar_id.split("/")[-1]
        print(jar_id)
        params = {
            "entry-class": class_name
        }
        if args is not None:
            comma_separated_string = ','.join([f'{k},{v}' for k, v in args.items()])
            params["programArg"] = comma_separated_string
        res = requests.post(f"http://{local_address}:{local_port}/jars/{jar_id}/run", 
                                params=params,
                            )
        print(res.json())
        return(res.json()["jobid"])
def stop_job(job_id):
    with en.G5kTunnel(roles["control"][0].address, 8081) as (local_address, local_port, _):
        requests.patch(f"http://{local_address}:{local_port}/jobs/{job_id}")
                   


## Deployment Flink

In [12]:
def deploy_stack(compose_file, env):
    with en.play_on(roles=roles, pattern_hosts=pattern_manager) as t:
        t.copy(src=f"../../envoy-passthrough.yml", dest="./")
        t.copy(src=f"../../envoy-alloy.yml", dest="./")
        t.copy(src=f"../../envoy-alloy-swarm.yml", dest="./")


    deploy_swarm(compose_file=compose_file, env=env)
    
def deploy_stack_and_publish(compose_file, target_rps, nb_sources=2, nb_partitions=2, nb_tm=3, nb_ts=1, nb_cpu=2, filters_data="", envoy_version="v1.29-alloy-nolog", partition_criteria="label", projection_criteria="", debug=False):
    env = {
        "NUM_SOURCES": nb_sources,
        "NUM_PARTITIONS": nb_partitions,
        "NUM_TM": nb_tm,
        "NUM_TS": nb_ts,
        "NUM_CPU": nb_ts,
        "FILTERS_DATA": filters_data,
        "ENVOY_VERSION": envoy_version,
        "PARTITION_CRITERIA": partition_criteria,
        "PROJECTION_CRITERIA": projection_criteria, 
    }
    if debug:
        env["DEBUG"] = "rest.flamegraph.enabled: true"
    deploy_stack(compose_file, env)
    stop_publisher()
    launch_publisher(target_rps=target_rps)    

In [13]:
def launch_run(job_name=None, jar_name="../../target/alloy-flink-tests-0.1.jar", class_name="TestGroupBy", job_params={}, duration=120):
    now_str = datetime.datetime.now().strftime('%Y%m%d%H%M%S') 
    if "--jobName" not in job_params:
        job_name = f"undefined-{class_name}"
    else:
        job_name = job_params["--jobName"]
    job_name = f"{job_name}_{now_str}"
    job_id = deploy_job(jar_name, f"be.uclouvain.gepiciad.alloy.{class_name}", job_params)
    
    start = datetime.datetime.now(tz=datetime.timezone.utc)
    print(f"Sleeping for {duration}s")
    sleep(duration)
    stop_job(job_id)
    end =  datetime.datetime.now(tz=datetime.timezone.utc)
    df_metric = get_metrics_tunnel(start, end, metric_query='sum(irate(container_cpu_usage_seconds_total{image!="", name=~"test_task.*|test_envoy.*|test_kafka.*"}[1m])) by (name)')
    df_metric["job_id"] = job_id
    df_metric["job_name"] = job_name
    df_metric.index = df_metric.index - df_metric.index.min()
    return df_metric

_*Prom requests*_

sum(irate(container_cpu_usage_seconds_total{image!="", name=~"test_task.*"}[1m])) by (name)

sum(irate(container_network_receive_bytes_total{image!="", name=~"test_task.*"}[1m])) by (name)

sum(irate(container_network_transmit_bytes_total{image!="", name=~"test_task.*"}[1m])) by (name)

sum(irate(flink_taskmanager_job_task_numRecordsOut[30s]))

flink_taskmanager_job_task_operator_KafkaSourceReader_KafkaConsumer_records_consumed_rate
flink_taskmanager_job_task_operator_KafkaProducer_record_send_rate

## Debugging

flink_taskmanager_job_task_operator_KafkaSourceReader_KafkaConsumer_records_per_request_avg
flink_taskmanager_job_task_operator_KafkaProducer_records_per_request_avg

flink_taskmanager_job_task_operator_KafkaSourceReader_KafkaConsumer_incoming_byte_rate
flink_taskmanager_job_task_operator_KafkaProducer_outgoing_byte_rate

## Deploy Alloy

In [47]:
compose_file = "docker-compose.yml"
compose_file = "docker-compose-envoy.yml"
compose_file = "docker-compose-alloy-test.yml"
df_metric = None
parallelism = 2
duration=300
deploy_stack_and_publish(compose_file, 5000, nb_sources=parallelism, nb_partitions=2, nb_tm=2)

Output()

Finished 1 tasks (copy) on {'gros-52.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

Cleaning old swarm deployment test
Launching new swarm deployment test: NUM_SOURCES='2' NUM_PARTITIONS='2' NUM_TM='2' docker stack deploy --compose-file ./docker-compose-alloy-test.yml test


...

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

HOST=kafka-edge1 PORT=9092 THROUGHPUT=5000 TOPIC=source REPLICAS=1 KEY_NAME=id KEY_VALUE=identifier docker stack deploy --compose-file ./docker-compose.berserker.yml test


Finished 2 tasks (shell,copy) on {'gros-52.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

## Group BY

### test regular groupby

In [48]:

class_name = "TestGroupBy"
params = {
    "--jobName" : f"{class_name}_{parallelism}",
    "--parallelism": f"{parallelism}",
    "--sourceTopic": "source"
}
df = launch_run("alloy", class_name=class_name, job_params=params, duration=duration)
df = df[df.index > pd.Timedelta(seconds=30)]
if df_metric is None:
    df_metric = df
else:
    df_metric = pd.concat([df_metric, df])

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=79032;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=215030;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=925042;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=542734;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

127.0.0.1:46675
4fa7c86f-916a-4ace-9c0e-21d09dcb33d3_alloy-flink-tests-0.1.jar
{'jobid': 'd5f14687d839dabde56e0bb7243dbe43'}


INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=336542;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=200606;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=156602;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=822582;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=667711;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=264357;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=351120;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=117463;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

2024-04-08T08:03:15+00:00Z-2024-04-08T08:08:16+00:00Z


### test alloy

In [49]:
sleep(30)
class_name = "TestGroupByReduced"
kafka_address = "envoy1:9093" # "kafka"
params = {
    "--kafkaAddress" : f"{kafka_address}",
    "--jobName" : f"{class_name}_{parallelism}",
    "--parallelism": f"{parallelism}",
    "--sourceTopic": "source"
}
df = launch_run("alloy", class_name=class_name, job_params=params, duration=duration)
df = df[df.index > pd.Timedelta(seconds=30)]
if df_metric is None:
    df_metric = df
else:
    df_metric = pd.concat([df_metric, df])

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=756972;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=65595;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=954922;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=328969;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

127.0.0.1:42931
28809b00-979d-497e-ac87-a3db5b591f74_alloy-flink-tests-0.1.jar
{'jobid': '3e1c1c12962aeaef7608399f9b03cb74'}


INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=675109;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=576681;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=966181;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=311858;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=22371;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=53152;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=33510;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=915351;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

2024-04-08T08:08:48+00:00Z-2024-04-08T08:13:49+00:00Z


In [51]:
df_metric["cpu_task_all"] = df_metric.filter(regex='cpu\_task.*').sum(axis=1)
df_metric["cpu_proxy_all"] = df_metric.filter(regex='cpu\_envoy.*').sum(axis=1)
df_metric["cpu_kafka_all"] = df_metric.filter(regex='cpu\_kafka.*').sum(axis=1)
display(df_metric[["job_name", "cpu_task_all", "cpu_proxy_all", "cpu_kafka_all"]].groupby(["job_name"]).mean())
display(df_metric[["job_name", "cpu_task_all", "cpu_proxy_all", "cpu_kafka_all"]].groupby(["job_name"]).mean().sum(axis=1))

display(df_metric[["job_name", "cpu_task_all", "cpu_proxy_all"]].groupby(["job_name"]).mean())
display(df_metric[["job_name", "cpu_task_all", "cpu_proxy_all"]].groupby(["job_name"]).mean().sum(axis=1))


,cpu_task_all,cpu_proxy_all,cpu_kafka_all
job_name,,,
TestGroupByReduced_2_20240408100847,1.368864,0.507059,1.926957
TestGroupBy_2_20240408100311,1.704398,0.006313,1.901032


job_name
TestGroupByReduced_2_20240408100847    3.802880
TestGroupBy_2_20240408100311           3.611743
dtype: float64

,cpu_task_all,cpu_proxy_all
job_name,,
TestGroupByReduced_2_20240408100847,1.368864,0.507059
TestGroupBy_2_20240408100311,1.704398,0.006313


job_name
TestGroupByReduced_2_20240408100847    1.875923
TestGroupBy_2_20240408100311           1.710711
dtype: float64

### test origin query with envoy (for passthrough)

In [96]:
sleep(30)
class_name = "TestGroupBy"
kafka_address = "envoy1:9093" # "kafka"
params = {
    "--kafkaAddress" : f"{kafka_address}",
    "--jobName" : f"{class_name}_{parallelism}",
    "--parallelism": f"{parallelism}",
    "--sourceTopic": "source"
}
df = launch_run("alloy", class_name=class_name, job_params=params)
df = df[df.index > pd.Timedelta(seconds=30)]
if df_metric is None:
    df_metric = df
else:
    df_metric = pd.concat([df_metric, df])



INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=826554;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py\transport.py]8;;\:]8;id=495146;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py#1874\1874]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=824876;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py\transport.py]8;;\:]8;id=779125;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py#1874\1874]8;;\

0.0.0.0:36807
0c7a720d-df5a-49ec-9ebe-e79a24450635_alloy-flink-tests-0.1.jar
{'jobid': 'b5734ffbf617e333d3bb341c763727ec'}


INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=145954;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py\transport.py]8;;\:]8;id=167904;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py#1874\1874]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=414793;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py\transport.py]8;;\:]8;id=386265;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py#1874\1874]8;;\

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=95853;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py\transport.py]8;;\:]8;id=239060;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py#1874\1874]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=972911;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py\transport.py]8;;\:]8;id=977388;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py#1874\1874]8;;\

2024-03-20T19:34:45+00:00Z-2024-03-20T19:36:46+00:00Z


In [97]:
df_metric["cpu_task_all"] = df_metric.filter(regex='cpu\_task.*').sum(axis=1)
df_metric["cpu_proxy_all"] = df_metric.filter(regex='cpu\_envoy.*').sum(axis=1)
display(df_metric[["job_name", "cpu_task_all", "cpu_proxy_all"]].groupby(["job_name"]).mean())
display(df_metric[["job_name", "cpu_task_all", "cpu_proxy_all"]].groupby(["job_name"]).mean().sum(axis=1))


,cpu_task_all,cpu_proxy_all
job_name,,
TestGroupBy_2_20240320203441,0.960722,0.340835


job_name
TestGroupBy_2_20240320203441    1.301557
dtype: float64

## Shuffle No chain

In [19]:
compose_file = "docker-compose.yml"
compose_file = "docker-compose-envoy.yml"
compose_file = "docker-compose-alloy-test-nochain.yml"
envoy_version = "v1.29-alloy-nolog"
#envoy_version = "v1.29-alloy"
df_metric = None
parallelism = 2
nb_partitions = 2
duration=300
max_quantity=0
throughput=10000
#throughput=1000
filters_data = ""
partition_criteria = "label"
nb_tm=2
nb_ts=3
nb_cpu = nb_ts + 1
fetchMinBytes = 100000
debug=True
filters_data_sql = ""
deploy_stack_and_publish(compose_file, throughput, nb_sources=parallelism, nb_partitions=nb_partitions, nb_tm=nb_tm, nb_ts=nb_ts, nb_cpu=nb_cpu, filters_data=filters_data, envoy_version=envoy_version, partition_criteria=partition_criteria,debug=debug)

Output()

Finished 1 tasks (copy) on {'gros-22.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

Cleaning old swarm deployment test
Launching new swarm deployment test: NUM_SOURCES='2' NUM_PARTITIONS='2' NUM_TM='2' NUM_TS='3' NUM_CPU='3' FILTERS_DATA='' ENVOY_VERSION='v1.29-alloy-nolog' PARTITION_CRITERIA='label' PROJECTION_CRITERIA='' DEBUG='rest.flamegraph.enabled: true' docker stack deploy --compose-file ./docker-compose-alloy-test-nochain.yml test


...

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

HOST=kafka-edge1 PORT=9092 THROUGHPUT=10000 TOPIC=source REPLICAS=1 KEY_NAME=id KEY_VALUE=identifier docker stack deploy --compose-file ./docker-compose.berserker.yml test


Finished 2 tasks (shell,copy) on {'gros-22.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

In [20]:
class_name = "shuffle.TestRawNoChain"
params = {
    "--jobName" : f"{class_name}_{parallelism}_raw",
    "--parallelism": f"{parallelism}",
    "--sourceTopic": "source",
    "--partitionCriteria": "label",
    "--filterData": filters_data_sql,
    "--fetchMinBytes": f"{fetchMinBytes}",
}
df = launch_run("alloy", class_name=class_name, job_params=params, duration=duration)
df = df[df.index > pd.Timedelta(seconds=30)]
if df_metric is None:
    df_metric = df
else:
    df_metric = pd.concat([df_metric, df])

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=673446;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=671636;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=838801;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=381037;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

127.0.0.1:44895
d9981855-42f7-4385-918b-474dd4f974e6_alloy-flink-tests-0.1.jar
{'jobid': 'aa799b9c2a73af64fa82aa074498bb5c'}
Sleeping for 300s


INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=698000;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=955135;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=632461;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=432500;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=519780;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=345297;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=456401;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=228089;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

2024-04-24T14:30:07+00:00Z-2024-04-24T14:35:08+00:00Z


In [21]:
fetchMinBytes = 300000

In [22]:
sleep(30)
class_name = "shuffle.TestReinterpretNoChain"
kafka_address = "envoy1:9093" # "kafka"
params = {
    "--kafkaAddress" : f"{kafka_address}",
    "--jobName" : f"{class_name}_{parallelism}_alloy",
    "--parallelism": f"{parallelism}",
    "--sourceTopic": "source",
    "--partitionCriteria": "label",
    "--fetchMinBytes": f"{fetchMinBytes}",
}
df = launch_run("alloy", class_name=class_name, job_params=params, duration=duration)
df = df[df.index > pd.Timedelta(seconds=30)]
if df_metric is None:
    df_metric = df
else:
    df_metric = pd.concat([df_metric, df])

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=734142;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=488310;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=457912;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=547433;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

127.0.0.1:35073
58ad1594-5773-4f03-bb8a-ae5192424705_alloy-flink-tests-0.1.jar
{'jobid': '817695f31c67dbe064f2f605ac44e192'}
Sleeping for 300s


INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=188945;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=949303;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=111415;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=793418;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=680340;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=141967;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=737390;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=776809;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

2024-04-24T14:35:40+00:00Z-2024-04-24T14:40:41+00:00Z


## Shuffle

In [24]:
compose_file = "docker-compose.yml"
compose_file = "docker-compose-envoy.yml"
compose_file = "docker-compose-alloy-test-nochain.yml"
compose_file = "docker-compose-alloy-test.yml"
envoy_version = "v1.29-alloy-nolog"
#envoy_version = "v1.29-alloy"
df_metric = None
parallelism = 2
nb_partitions = 2
duration=120
max_quantity=0
throughput=10000
#throughput=1000
filters_data = ""
partition_criteria = "label"
nb_tm=2
nb_ts=1
nb_cpu = nb_ts + 1
fetchMinBytes = 100000
debug=True
filters_data_sql = ""
deploy_stack_and_publish(compose_file, throughput, nb_sources=parallelism, nb_partitions=nb_partitions, nb_tm=nb_tm, nb_ts=nb_ts, nb_cpu=nb_cpu, filters_data=filters_data, envoy_version=envoy_version, partition_criteria=partition_criteria,debug=debug)

Output()

Finished 1 tasks (copy) on {'gros-22.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

Cleaning old swarm deployment test
Launching new swarm deployment test: NUM_SOURCES='2' NUM_PARTITIONS='2' NUM_TM='2' NUM_TS='1' NUM_CPU='1' FILTERS_DATA='' ENVOY_VERSION='v1.29-alloy-nolog' PARTITION_CRITERIA='label' PROJECTION_CRITERIA='' DEBUG='rest.flamegraph.enabled: true' docker stack deploy --compose-file ./docker-compose-alloy-test.yml test


...

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

HOST=kafka-edge1 PORT=9092 THROUGHPUT=10000 TOPIC=source REPLICAS=1 KEY_NAME=id KEY_VALUE=identifier docker stack deploy --compose-file ./docker-compose.berserker.yml test


Finished 2 tasks (shell,copy) on {'gros-22.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

In [25]:
duration=300

In [26]:
class_name = "shuffle.TestRaw"
params = {
    "--jobName" : f"{class_name}_{parallelism}_raw",
    "--parallelism": f"{parallelism}",
    "--sourceTopic": "source",
    "--partitionCriteria": "label",
    "--filterData": filters_data_sql,
    "--fetchMinBytes": f"{fetchMinBytes}",
}
df = launch_run("alloy", class_name=class_name, job_params=params, duration=duration)
df = df[df.index > pd.Timedelta(seconds=30)]
if df_metric is None:
    df_metric = df
else:
    df_metric = pd.concat([df_metric, df])

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=182732;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=703870;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=30373;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=405027;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

127.0.0.1:46251
7055470a-38ae-43dc-bbac-a6be7f98d2fa_alloy-flink-tests-0.1.jar
{'jobid': '7eb24c9f28d5a985078461d0780581d1'}
Sleeping for 300s


INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=465421;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=695318;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=534161;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=479785;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=816146;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=899430;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=771292;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=528815;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

2024-04-24T14:53:18+00:00Z-2024-04-24T14:58:18+00:00Z


In [28]:
fetchMinBytes = 300000

In [29]:
sleep(30)
class_name = "shuffle.TestReinterpret"
kafka_address = "envoy1:9093" # "kafka"
params = {
    "--kafkaAddress" : f"{kafka_address}",
    "--jobName" : f"{class_name}_{parallelism}_alloy",
    "--parallelism": f"{parallelism}",
    "--sourceTopic": "source",
    "--partitionCriteria": "label",
    "--fetchMinBytes": f"{fetchMinBytes}",
}
df = launch_run("alloy", class_name=class_name, job_params=params, duration=duration)
df = df[df.index > pd.Timedelta(seconds=30)]
if df_metric is None:
    df_metric = df
else:
    df_metric = pd.concat([df_metric, df])


INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=792765;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=877357;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=757027;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=147394;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

127.0.0.1:35857
758d266a-9313-4de2-b289-ba2a6996e9fd_alloy-flink-tests-0.1.jar
{'jobid': 'fe2a1c2600ffd2fa2c1a0c259fa86fae'}
Sleeping for 300s


INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=219936;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=290211;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=259634;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=667483;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=539973;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=977133;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=413441;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=750772;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

2024-04-24T16:11:50+00:00Z-2024-04-24T16:16:51+00:00Z


In [30]:
df_metric["cpu_tm_all"] = df_metric.filter(regex='cpu\_task.*').sum(axis=1)
df_metric["cpu_proxy_all"] = df_metric.filter(regex='cpu\_envoy.*').sum(axis=1)
display(df_metric[["job_name", "cpu_tm_all", "cpu_proxy_all"]].groupby(["job_name"]).mean())
display(df_metric[["job_name", "cpu_tm_all", "cpu_proxy_all"]].groupby(["job_name"]).mean().sum(axis=1))


,cpu_tm_all,cpu_proxy_all
job_name,,
shuffle.TestRaw_2_raw_20240424165314,0.320085,0.006915
shuffle.TestReinterpret_2_alloy_20240424181149,0.253945,0.070136


job_name
shuffle.TestRaw_2_raw_20240424165314              0.327000
shuffle.TestReinterpret_2_alloy_20240424181149    0.324082
dtype: float64

### shuffle passthrough

In [66]:
sleep(30)
kafka_address = "envoy1:9096" # "kafka"
class_name = "shuffle.TestRaw"
params = {
    "--jobName" : f"{class_name}_{parallelism}_passthrough",
    "--parallelism": f"{parallelism}",
    "--sourceTopic": "source",
    "--kafkaAddress" : f"{kafka_address}",    
    "--maxQuantity": max_quantity
}
df = launch_run("passthrough", class_name=class_name, job_params=params, duration=duration)
df = df[df.index > pd.Timedelta(seconds=30)]
if df_metric is None:
    df_metric = df
else:
    df_metric = pd.concat([df_metric, df])

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=463082;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=731497;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=74886;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=337791;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

127.0.0.1:40365
921ae425-9354-4df8-8c04-c44795c7b2c3_alloy-flink-tests-0.1.jar
{'jobid': 'dfe1337b4e09c3a57ebe44571bd8a463'}


INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=220881;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=862155;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=369089;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=798537;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=858163;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=426211;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=659553;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=7044;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

2024-04-09T14:07:07+00:00Z-2024-04-09T14:09:07+00:00Z


In [67]:
df_metric["cpu_tm_all"] = df_metric.filter(regex='cpu\_task.*').sum(axis=1)
df_metric["cpu_proxy_all"] = df_metric.filter(regex='cpu\_envoy.*').sum(axis=1)
display(df_metric[["job_name", "cpu_tm_all", "cpu_proxy_all"]].groupby(["job_name"]).mean())
display(df_metric[["job_name", "cpu_tm_all", "cpu_proxy_all"]].groupby(["job_name"]).mean().sum(axis=1))


,cpu_tm_all,cpu_proxy_all
job_name,,
shuffle.TestRaw_4_passthrough_20240409160705,1.082697,0.392703
shuffle.TestRaw_4_raw-alloy_20240409160433,0.292995,0.026204
shuffle.TestRaw_4_raw_20240409155925,1.164166,0.006421
shuffle.TestReinterpret_4_alloy_20240409160200,1.119515,0.621638


job_name
shuffle.TestRaw_4_passthrough_20240409160705      1.475400
shuffle.TestRaw_4_raw-alloy_20240409160433        0.319200
shuffle.TestRaw_4_raw_20240409155925              1.170587
shuffle.TestReinterpret_4_alloy_20240409160200    1.741153
dtype: float64

## Projection

In [27]:
compose_file = "docker-compose.yml"
compose_file = "docker-compose-envoy.yml"
compose_file = "docker-compose-alloy-test-nochain.yml"
envoy_version = "v1.29-alloy-nolog"
#envoy_version = "v1.29-alloy"
df_metric = None
parallelism = 2
nb_partitions = 2
duration=120
max_quantity=0
throughput=100000
#throughput=1000
filters_data = ""
partition_criteria = ""
projection_criteria = "id,ts"
nb_tm=2
nb_ts=3
nb_cpu = nb_ts + 1
fetchMinBytes = 1000000
debug=True
filters_data_sql = ""
deploy_stack_and_publish(compose_file, throughput, nb_sources=parallelism, nb_partitions=nb_partitions, nb_tm=nb_tm, nb_ts=nb_ts, nb_cpu=nb_cpu, filters_data=filters_data, envoy_version=envoy_version, partition_criteria=partition_criteria, projection_criteria=projection_criteria , debug=debug)

Output()

Finished 1 tasks (copy) on {'gros-49.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

Cleaning old swarm deployment test
Launching new swarm deployment test: NUM_SOURCES='2' NUM_PARTITIONS='2' NUM_TM='2' NUM_TS='3' NUM_CPU='3' FILTERS_DATA='' ENVOY_VERSION='v1.29-alloy-nolog' PARTITION_CRITERIA='' PROJECTION_CRITERIA='id,ts' DEBUG='rest.flamegraph.enabled: true' docker stack deploy --compose-file ./docker-compose-alloy-test-nochain.yml test


...

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

HOST=kafka-edge1 PORT=9092 THROUGHPUT=100000 TOPIC=source REPLICAS=1 KEY_NAME=id KEY_VALUE=identifier docker stack deploy --compose-file ./docker-compose.berserker.yml test


Finished 2 tasks (shell,copy) on {'gros-49.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

In [33]:
class_name = "projection.TestRawNoChain"
params = {
    "--jobName" : f"{class_name}_{parallelism}_raw",
    "--parallelism": f"{parallelism}",
    "--sourceTopic": "source",
    "--partitionCriteria": "id",
    "--filterData": filters_data_sql,
    "--fetchMinBytes": f"{fetchMinBytes}",
}
df = launch_run("alloy", class_name=class_name, job_params=params, duration=duration)
df = df[df.index > pd.Timedelta(seconds=30)]
if df_metric is None:
    df_metric = df
else:
    df_metric = pd.concat([df_metric, df])

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=919073;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=845880;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=703905;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=361363;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

127.0.0.1:33019
ee376e1e-a814-41fd-abba-617725ccea1e_alloy-flink-tests-0.1.jar
{'jobid': '55f9f578f34432450e8ecab80ca36ae0'}
Sleeping for 120s


INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=9494;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=226311;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=696442;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=782907;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=957447;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=671628;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=705574;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=825716;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

2024-04-23T15:17:13+00:00Z-2024-04-23T15:19:14+00:00Z


In [32]:
sleep(30)
class_name = "projection.TestReinterpretNoChain"
kafka_address = "envoy1:9093" # "kafka"
params = {
    "--kafkaAddress" : f"{kafka_address}",
    "--jobName" : f"{class_name}_{parallelism}_alloy",
    "--parallelism": f"{parallelism}",
    "--sourceTopic": "source",
    "--partitionCriteria": "id",
    "--fetchMinBytes": f"{fetchMinBytes}",
}
df = launch_run("alloy", class_name=class_name, job_params=params, duration=duration)
df = df[df.index > pd.Timedelta(seconds=30)]
if df_metric is None:
    df_metric = df
else:
    df_metric = pd.concat([df_metric, df])

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=33716;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=989198;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=384019;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=607143;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

127.0.0.1:34367
dc774c5a-3c5a-43e1-b10a-fc7b49c42776_alloy-flink-tests-0.1.jar
{'jobid': 'd5de5cf2d05f50ae8dc4f3f6115235b8'}
Sleeping for 120s


INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=376271;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=689477;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=74887;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=915830;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=925563;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=952815;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=903080;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=205214;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

2024-04-23T15:13:13+00:00Z-2024-04-23T15:15:13+00:00Z


## Filter

In [193]:
compose_file = "docker-compose.yml"
compose_file = "docker-compose-envoy.yml"
compose_file = "docker-compose-alloy-test-nochain.yml"
envoy_version = "v1.29-alloy-nolog"
#envoy_version = "v1.29-alloy"
df_metric = None
parallelism = 2
nb_partitions = 2
duration=120
max_quantity=0
throughput=100000
#throughput=1000
filters_data = "filter_data,^0.*"
partition_criteria = ""
nb_tm=2
nb_ts=3
nb_cpu = nb_ts + 1
fetchMinBytes = 1000000
debug=True
filters_data_sql = "WHERE filter_data LIKE '0%'"
deploy_stack_and_publish(compose_file, throughput, nb_sources=parallelism, nb_partitions=nb_partitions, nb_tm=nb_tm, nb_ts=nb_ts, nb_cpu=nb_cpu, filters_data=filters_data, envoy_version=envoy_version, partition_criteria=partition_criteria,debug=debug)

Output()

Finished 1 tasks (copy) on {'gros-122.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

Cleaning old swarm deployment test
Launching new swarm deployment test: NUM_SOURCES='2' NUM_PARTITIONS='2' NUM_TM='2' NUM_TS='3' NUM_CPU='3' FILTERS_DATA='filter_data,^0.*' ENVOY_VERSION='v1.29-alloy-nolog' PARTITION_CRITERIA='' DEBUG='rest.flamegraph.enabled: true' docker stack deploy --compose-file ./docker-compose-alloy-test-nochain.yml test


...

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

HOST=kafka-edge1 PORT=9092 THROUGHPUT=100000 TOPIC=source REPLICAS=1 KEY_NAME=id KEY_VALUE=identifier docker stack deploy --compose-file ./docker-compose.berserker.yml test


Finished 2 tasks (shell,copy) on {'gros-122.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

In [194]:
duration=300

In [195]:
class_name = "shuffle.TestRawNoChain"
params = {
    "--jobName" : f"{class_name}_{parallelism}_raw",
    "--parallelism": f"{parallelism}",
    "--sourceTopic": "source",
    "--partitionCriteria": "id",
    "--filterData": filters_data_sql,
    "--fetchMinBytes": f"{fetchMinBytes}",
}
df = launch_run("alloy", class_name=class_name, job_params=params, duration=duration)
df = df[df.index > pd.Timedelta(seconds=30)]
if df_metric is None:
    df_metric = df
else:
    df_metric = pd.concat([df_metric, df])

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=293047;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=91559;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=443426;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=325536;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

127.0.0.1:45621
26fb8556-33b8-4689-a328-a23776c6a0a7_alloy-flink-tests-0.1.jar
{'jobid': 'a7ad22e8eff86b8f6113062d797d2f77'}
Sleeping for 300s


INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=880335;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=804310;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=95398;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=296270;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=50238;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=92390;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=40580;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=761189;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

2024-04-22T22:08:29+00:00Z-2024-04-22T22:13:30+00:00Z


In [196]:
sleep(30)
class_name = "shuffle.TestReinterpretNoChain"
kafka_address = "envoy1:9093" # "kafka"
params = {
    "--kafkaAddress" : f"{kafka_address}",
    "--jobName" : f"{class_name}_{parallelism}_alloy",
    "--parallelism": f"{parallelism}",
    "--sourceTopic": "source",
    "--partitionCriteria": "id",
    "--fetchMinBytes": f"{fetchMinBytes}",
}
df = launch_run("alloy", class_name=class_name, job_params=params, duration=duration)
df = df[df.index > pd.Timedelta(seconds=30)]
if df_metric is None:
    df_metric = df
else:
    df_metric = pd.concat([df_metric, df])


INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=261528;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=793701;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=752293;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=487806;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

127.0.0.1:33923
459c6d12-9f53-4d65-a3c1-7b40a4ae675c_alloy-flink-tests-0.1.jar
{'jobid': '6eed963e7cb0f4778576e622878be0ba'}
Sleeping for 300s


INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=664507;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=896548;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=568455;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=940063;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=212897;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=412606;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=955737;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=909332;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

2024-04-22T22:14:05+00:00Z-2024-04-22T22:19:05+00:00Z


In [73]:
class_name = "shuffle.TestKeyBy"
kafka_address = "envoy1:9093" # "kafka"
params = {
    "--kafkaAddress" : f"{kafka_address}",
    "--jobName" : f"{class_name}_{parallelism}_envoy",
    "--parallelism": f"{parallelism}",
    "--sourceTopic": "source",
    "--filterData": filters_data_sql,
    "--partitionCriteria": "id",
}
df = launch_run("alloy", class_name=class_name, job_params=params, duration=duration)
df = df[df.index > pd.Timedelta(seconds=30)]
if df_metric is None:
    df_metric = df
else:
    df_metric = pd.concat([df_metric, df])

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=392465;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=473376;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=235591;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=894118;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

127.0.0.1:34731
5ec1e2db-0acf-4b8a-9833-ace4f413e805_alloy-flink-tests-0.1.jar
{'jobid': 'afa7de018797748275f6000c9dc2a829'}
Sleeping for 120s


INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=949713;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=872066;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=53828;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=482554;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=43089;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=219952;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=300457;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=404370;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

2024-04-20T11:26:16+00:00Z-2024-04-20T11:28:18+00:00Z


In [74]:
class_name = "shuffle.TestRaw"
kafka_address = "envoy1:9096" # "kafka"
params = {
    "--kafkaAddress" : f"{kafka_address}",
    "--jobName" : f"{class_name}_{parallelism}_passthrough",
    "--parallelism": f"{parallelism}",
    "--sourceTopic": "source",
    "--filterData": filters_data_sql,
    "--partitionCriteria": partition_criteria,
}
df = launch_run("alloy", class_name=class_name, job_params=params, duration=duration)
df = df[df.index > pd.Timedelta(seconds=30)]
if df_metric is None:
    df_metric = df
else:
    df_metric = pd.concat([df_metric, df])

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=26506;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=269215;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=295328;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=286313;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

127.0.0.1:45353


ERROR    Secsh channel 0 open FAILED: Connection refused: Connect failed    ]8;id=465508;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py\transport.py]8;;\:]8;id=191987;file:///usr/local/lib/python3.10/dist-packages/paramiko/transport.py#1909\1909]8;;\

2024-04-21 16:51:30,434| ERROR   | Could not establish connection from local ('127.0.0.1', 45353) to remote ('gros-44.nancy.grid5000.fr', 8081) side of the tunnel: open new channel ssh error: ChannelException(2, 'Connect failed')


ERROR    Could not establish connection from local ('127.0.0.1', 45353) to   ]8;id=594937;file:///usr/local/lib/python3.10/dist-packages/sshtunnel.py\sshtunnel.py]8;;\:]8;id=194611;file:///usr/local/lib/python3.10/dist-packages/sshtunnel.py#394\394]8;;\
         remote ('gros-44.nancy.grid5000.fr', 8081) side of the tunnel: open                 
         new channel ssh error: ChannelException(2, 'Connect failed')                        

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [ ]:
df_metric["cpu_tm_all"] = df_metric.filter(regex='cpu\_task.*').sum(axis=1)
df_metric["cpu_proxy_all"] = df_metric.filter(regex='cpu\_envoy.*').sum(axis=1)
display(df_metric[["job_name", "cpu_tm_all", "cpu_proxy_all"]].groupby(["job_name"]).mean())
display(df_metric[["job_name", "cpu_tm_all", "cpu_proxy_all"]].groupby(["job_name"]).mean().sum(axis=1))


,cpu_tm_all,cpu_proxy_all
job_name,,
shuffle.TestRaw_2_raw_20240418220751,0.758115,0.006337
shuffle.TestRaw_2_raw_20240418221145,0.696712,0.006653
shuffle.TestReinterpret_2_alloy_20240418221739,0.958033,0.783462


job_name
shuffle.TestRaw_2_raw_20240418220751              0.764452
shuffle.TestRaw_2_raw_20240418221145              0.703365
shuffle.TestReinterpret_2_alloy_20240418221739    1.741495
dtype: float64

## direct keyby

In [ ]:
parallelism = 2
class_name = "TestGroupByReducedKeyBy"
params = {
    #"--kafkaAddress" : f"{kafka_address}",
    "--jobName" : f"{class_name}_{parallelism}",
    "--parallelism": f"{parallelism}",
    "--sourceTopic": "source"
}

job_id = deploy_job("../../target/alloy-flink-tests-0.1.jar", f"be.uclouvain.gepiciad.alloy.{class_name}", params)

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=910759;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py\transport.py]8;;\:]8;id=571583;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py#1874\1874]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=922346;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py\transport.py]8;;\:]8;id=727094;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py#1874\1874]8;;\

0.0.0.0:41263
b1598e0c-dc2f-408d-ab6c-b79ed5e7332c_alloy-flink-tests-0.1.jar
{'jobid': '9f5d7863c69aeb2fa4908a35efbb1dfb'}


## Test reduced

In [ ]:
job_id = deploy_job("../../target/alloy-flink-tests-0.1.jar", "be.uclouvain.gepiciad.alloy.TestGroupBySqlReduce")

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=941096;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py\transport.py]8;;\:]8;id=297575;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py#1874\1874]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=999130;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py\transport.py]8;;\:]8;id=164478;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py#1874\1874]8;;\

0.0.0.0:35077
7d0a7a9d-7f43-4103-a5d7-4fdc8ce030bf_alloy-flink-tests-0.1.jar
{'jobid': '3b8239058b59a8db52f2fe0fc76581cd'}


stop_publisher()
launch_publisher(target_rps=10000, topic="reduced", key_name="label", key_value="classifier")

In [ ]:
parallelism = 2
class_name = "TestGroupByReduced"
kafka_address = "envoy1:9093" # "kafka"
params = {
    "--kafkaAddress" : f"{kafka_address}",
    "--jobName" : f"{class_name}_{parallelism}",
    "--parallelism": f"{parallelism}",
    "--sourceTopic": "sql-reduced"
}

job_id = deploy_job("../../target/alloy-flink-tests-0.1.jar", f"be.uclouvain.gepiciad.alloy.{class_name}", params)


INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=238327;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py\transport.py]8;;\:]8;id=246660;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py#1874\1874]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=244036;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py\transport.py]8;;\:]8;id=229427;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py#1874\1874]8;;\

0.0.0.0:37449
515843da-094c-4afa-bc51-adc54ad88184_alloy-flink-tests-0.1.jar
{'jobid': '41a064cccc5db3f75cb85b18606a92de'}


In [ ]:
parallelism = 2
class_name = "TestGroupByReducedKeyBy"
kafka_address = "envoy1:9093" # "kafka"
params = {
    "--kafkaAddress" : f"{kafka_address}",
    "--jobName" : f"{class_name}_{parallelism}",
    "--parallelism": f"{parallelism}",
    "--sourceTopic": "sql-reduced"
}

job_id = deploy_job("../../target/alloy-flink-tests-0.1.jar", f"be.uclouvain.gepiciad.alloy.{class_name}", params)

INFO     Connected (version 2.0, client OpenSSH_8.4p1)                      ]8;id=140311;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py\transport.py]8;;\:]8;id=11440;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py#1874\1874]8;;\

INFO     Authentication (publickey) successful!                             ]8;id=251896;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py\transport.py]8;;\:]8;id=277581;file:///home/master/.local/lib/python3.7/site-packages/paramiko/transport.py#1874\1874]8;;\

0.0.0.0:33389
2c73c870-4b7e-468e-b874-0e4310134221_alloy-flink-tests-0.1.jar
{'jobid': '35c40da04074b2d66b247dd497893c88'}
